Painel de Leitos SUS

Graficos e plano de acao

In [ ]:
import os
import subprocess
import sys

repo_url = "https://github.com/GabrielSantiago-TI/bigdata_leitos.git"
repo_dir = "/content/bigdata_leitos"

if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", repo_url, repo_dir], check=True)

os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Projeto carregado no Colab")

In [ ]:
import importlib
import pandas as pd

import charts
import loaders
import simulation

importlib.reload(charts)
importlib.reload(loaders)
importlib.reload(simulation)

base, evolucao, competencia = loaders.carregar_base_app()

print(f"Competencia analisada: {competencia}")
print(f"Registros carregados: {len(base):,}".replace(",", "."))

Graficos

In [ ]:
charts.grafico_densidade(base).show()

In [ ]:
charts.grafico_barras(base).show()

In [ ]:
charts.grafico_setores(base).show()

In [ ]:
charts.grafico_evolucao(evolucao).show()

Plano de acao

In [ ]:
simulacao = simulation.gerar_status_leitos(
    simulation.calcular_ocupacao(base, cenario="Alta demanda")
)

plano_acao = (
    simulacao[
        [
            "regiao",
            "uf",
            "municipio",
            "nome_estabelecimento",
            "leitos_sus",
            "ocupacao_pct",
            "fila",
            "vagas_6h",
            "indice_pressao",
            "status",
            "acao",
        ]
    ]
    .rename(
        columns={
            "regiao": "Regiao",
            "uf": "UF",
            "municipio": "Municipio",
            "nome_estabelecimento": "Hospital",
            "leitos_sus": "Leitos SUS",
            "ocupacao_pct": "Ocupacao (%)",
            "fila": "Fila simulada",
            "vagas_6h": "Vagas em 6h",
            "indice_pressao": "Indice de pressao",
            "status": "Status",
            "acao": "Plano de acao",
        }
    )
)

plano_acao["Ocupacao (%)"] = plano_acao["Ocupacao (%)"].round(1)
plano_acao["Indice de pressao"] = plano_acao["Indice de pressao"].round(1)

plano_acao.head(25)

In [ ]:
resumo_plano = (
    plano_acao
    .groupby(["Status", "Plano de acao"], as_index=False)
    .agg(
        hospitais=("Hospital", "count"),
        leitos_sus=("Leitos SUS", "sum"),
        fila_simulada=("Fila simulada", "sum"),
        vagas_6h=("Vagas em 6h", "sum"),
        pressao_media=("Indice de pressao", "mean"),
    )
    .sort_values(["Status", "pressao_media"], ascending=[True, False])
)

resumo_plano["pressao_media"] = resumo_plano["pressao_media"].round(1)
resumo_plano

In [ ]:
plano_acao.to_csv("plano_acao_professor.csv", index=False, encoding="utf-8-sig")
resumo_plano.to_csv("resumo_plano_professor.csv", index=False, encoding="utf-8-sig")

print("Arquivos gerados:")
print("plano_acao_professor.csv")
print("resumo_plano_professor.csv")